In [8]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

data_dir = Path("../model_outputs")

ds_r1  = xr.open_dataset(data_dir / "tas_Amon_MPI-ESM1-2-LR_historical-ssp245_r1i1p1f1_g025_185001-210012.nc")
ds_r25 = xr.open_dataset(data_dir / "tas_Amon_MPI-ESM1-2-LR_historical-ssp245_r25i1p1f1_g025_185001-210012.nc")
ds_pic = xr.open_dataset(data_dir / "tas_Amon_MPI-ESM1-2-LR_piControl_r1i1p1f1_g025_185001-284912.nc")
lm     = xr.open_dataset(data_dir / "landmask.nc")

land_mask  = lm["sftlf"] > 0.5
ocean_mask = lm["sftlf"] <= 0.5

def global_mean(ds):
    weights = np.cos(np.deg2rad(ds.lat))
    return ds["tas"].weighted(weights).mean(dim=["lat", "lon"])

gm_r1  = global_mean(ds_r1)  - 273.15
gm_r25 = global_mean(ds_r25) - 273.15
gm_pic = global_mean(ds_pic) - 273.15

gm_r1_ann  = gm_r1.resample(time="YE").mean()
gm_r25_ann = gm_r25.resample(time="YE").mean()
gm_pic_ann = gm_pic.resample(time="YE").mean()

# Land / ocean means for piControl
tas_pic     = ds_pic["tas"]
lat_w_pic   = np.cos(np.deg2rad(ds_pic.lat))
w3d_pic     = lat_w_pic.broadcast_like(tas_pic)

def masked_annual_mean_pic(mask):
    w = w3d_pic.where(mask)
    return ((tas_pic.where(mask) * w).sum(dim=["lat", "lon"]) / w.sum(dim=["lat", "lon"]) - 273.15).resample(time="YE").mean()

land_pic  = masked_annual_mean_pic(land_mask)
ocean_pic = masked_annual_mean_pic(ocean_mask)

years_pic = gm_pic_ann.time.dt.year.values

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=False)

# --- Top panel: historical + SSP245 (r1 and r25) ---
ax = axes[0]
ax.plot(gm_r1_ann.time.dt.year,  gm_r1_ann.values,  color="steelblue", lw=1.2, label="r1  (ensemble member 1)")
ax.plot(gm_r25_ann.time.dt.year, gm_r25_ann.values, color="tomato",    lw=1.2, label="r25 (ensemble member 25)")
ax.axvline(2014, color="gray", lw=1, linestyle="--", label="historical → SSP2-4.5")
ax.set_title("Historical + SSP2-4.5 (1850–2100) — Global Mean Near-Surface Temperature")
ax.set_ylabel("Temperature (°C)")
ax.set_xlabel("Year")
ax.legend()
ax.grid(alpha=0.3)

# --- Bottom panel: piControl global / land / ocean ---
ax = axes[1]
ax.plot(years_pic, gm_pic_ann.values,  color="black",      lw=1.5, label="Global mean")
ax.plot(years_pic, ocean_pic.values,   color="steelblue",  lw=1.5, label="Ocean surface")
ax.plot(years_pic, land_pic.values,    color="seagreen",   lw=1.5, label="Land surface")
ax.set_title("MPI-ESM1-2-LR  |  piControl Annual Mean tas")
ax.set_ylabel("Temperature (°C)")
ax.set_xlabel("Model year")
ax.legend()
ax.grid(alpha=0.3)

fig.text(0.99, 0.01, "Nasrul Huda", ha="right", va="bottom", fontsize=9, color="gray")

plt.tight_layout()
plt.savefig("global_mean_tas_timeseries.pdf", format="pdf", bbox_inches="tight")
plt.show()